In [3]:
import sys
import torch
import transformers
import bitsandbytes as bnb

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

print(f"PyTorch CUDA:       {torch.version.cuda}")
print(f"Transformers:       {transformers.__version__}")
print(f"bitsandbytes:       {bnb.__version__}")


ImportError: libcusparseLt.so.0: cannot open shared object file: No such file or directory

In [ ]:

# 1. Define NF4 quantization config for QLoRA
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
print(f"\nLoading processor for {model_id}...")
processor = AutoProcessor.from_pretrained(model_id)


In [ ]:

print(f"Loading model in 4-bit...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)


In [ ]:

vram_gb = model.get_memory_footprint() / (1024**3)
vram_allocated_gb = torch.cuda.memory_allocated(0) / (1024**3)

print("\n" + "="*45)
print("✅ SUCCESS: Qwen 2.5 VL Loaded in 4-bit!")
print("="*45)
print(f"Model Memory Footprint: {vram_gb:.2f} GB")
print(f"Total VRAM Allocated:   {vram_allocated_gb:.2f} GB / 24.00 GB")
print("="*45)

In [ ]:
# Below are utility functions to unload and reload the model from GPU VRAM.

import gc
import torch

def force_unload(*objects_to_delete):
    """
    Aggressively purges passed objects (model, processor, trainer) from memory 
    and flushes the PyTorch CUDA cache.
    """
    print("Purging objects from VRAM...")
    
    # 1. Delete the specific objects passed to the function
    for obj in objects_to_delete:
        if obj is not None:
            del obj
            
    # 2. Run garbage collection twice (resolves circular references)
    gc.collect()
    gc.collect()
    
    # 3. Release PyTorch's cached/reserved GPU memory
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
    # 4. Print current memory state
    allocated = torch.cuda.memory_allocated(0) / (1024**3)
    reserved = torch.cuda.memory_reserved(0) / (1024**3)
    print(f"✅ VRAM Purge complete!")
    print(f"Currently Allocated VRAM: {allocated:.2f} GB")
    print(f"Currently Reserved VRAM:  {reserved:.2f} GB")

def reload_model(model_id="Qwen/Qwen2.5-VL-7B-Instruct", load_in_4bit=True):
    """
    Reloads the model into GPU VRAM using the previously defined loader.
    """
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    
    print(f"\nReloading {model_id} into VRAM...")
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["visual"]
    ) if load_in_4bit else None

    processor = AutoProcessor.from_pretrained(model_id)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    
    allocated = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"✅ Reload complete! Active VRAM: {allocated:.2f} GB")
    return model, processor

### Below script unloads the model

In [ ]:
# Pass ALL objects that touch the model
force_unload(model, processor, trainer)

# Erase the variables from the global notebook scope
model = None
processor = None
trainer = None

# Force a final cache clear just in case Jupyter held a hidden tensor
torch.cuda.empty_cache()

### belwo script reloads the model

In [ ]:
model, processor = reload_model("Qwen/Qwen2.5-VL-7B-Instruct", load_in_4bit=True)